# Pyke (Python Knowledge Engine)

A refresher on **Pyke** — a logic-programming / expert-system engine for Python that does
both **forward** and **backward chaining**, and can emit Python *plans* (compiled functions)
as the by-product of a proof.

**Domain:** Symbolic AI & Logic · **runnable:** yes · _Pyke itself is a Python-2-era SourceForge
project, so the live engine cells are gated; a tiny pure-Python chainer demonstrates the same idea._

## 1. What & Why

**Pyke** (the *Python Knowledge Engine*) is a knowledge-based inference engine. You write
**facts** and **rules** in Pyke's own `.kfb`/`.krb` files; Pyke *compiles* them into Python
modules and runs an inference engine that derives new facts (forward chaining) or proves
goals on demand (backward chaining). Unlike most rule engines, Pyke does **both** styles of
chaining and, uniquely, can assemble a **plan** — a callable Python function — as a side
effect of a successful backward-chaining proof. That makes it a tool for *program assembly*,
not just question answering.

**The problem it solves.** Some logic is naturally declarative — entitlements, configuration,
relationships, deductions over a knowledge base — and ages badly as nested `if` statements.
Pyke lets you state the rules once and ask the engine *what follows* or *prove this goal*. Its
headline trick, **plan generation**, targets a niche: "given these requirements, automatically
wire together the right sequence of Python calls." The proof of *how* a goal is satisfied
becomes runnable code you can call later without re-running inference.

**Reach for it when:**

- You want Prolog-style backward chaining **and** Rete-style forward chaining in one Python tool.
- You want the inference to *produce code* (a plan) you execute repeatedly — Pyke's distinctive use case.
- You're studying classic symbolic AI and want a Python-native engine to poke at.

**Don't reach for it when** you need an actively-maintained, pip-installable dependency for
production: Pyke's last release (1.1.1) predates Python 3's maturity and it is effectively
unmaintained. For new work prefer `swi-prolog`/pyswip (backward chaining), `clips`/clipspy or
`experta` (forward chaining), or `datalog` (pure deduction). Pyke is best understood as a
historically important, idea-rich engine — especially for the plan-generation concept.

## 2. Mental Model

Think of Pyke as a **compiler + two-direction prover** sitting over a knowledge base:

```
   .kfb (facts)        .krb (rules)
        \                 /
         \   pyke compiles to Python modules
          \             /
           v           v
        +-----------------------+
        |   knowledge_engine    |
        +-----------------------+
          |                   |
   activate(rb)          prove_goal(g)
   FORWARD chaining       BACKWARD chaining
   data -> derive ALL     goal -> search for
   consequences, store    a proof, bind vars
          |                   |
          v                   v
    new facts in KB      (vars, PLAN) — the plan
                          is callable Python code
```

Two directions, one engine:

- **Forward chaining (`fc_rule`):** start from facts, fire rules, `assert` new facts until
  nothing new appears (a fixpoint). Good for "compute everything that follows."
- **Backward chaining (`bc_rule`):** start from a goal, recursively try rules whose head
  unifies with it (like Prolog). Good for "is this true, and with what bindings?" — and only
  here can a **plan** be assembled.

The thing to internalize: Pyke **compiles** your `.krb`/`.kfb` text into `.py` files in a
`compiled_krb/` cache on first run, then imports and runs them. The DSL is *generated code*,
not interpreted on the fly.

## 3. Key Concepts

| Term | What it is |
|------|-----------|
| **Fact** | A ground statement in a knowledge base: `family.son_of(bruce, thomas)`. Stored in a `.kfb` file or asserted at runtime. |
| **`.kfb` (fact base)** | A file of plain facts under a named base, e.g. base `family` with lines like `son_of(bruce, thomas)`. |
| **`.krb` (rule base)** | A file of rules: forward-chaining (`fc_*`) and/or backward-chaining (`bc_*`). |
| **Pattern variable** | `$x` — a logic variable that **unifies** with fact terms; consistent bindings across patterns form a join. |
| **Forward rule (`fc_rule`)** | `foreach <premises> assert <conclusions>`. Fires on data, asserts derived facts. |
| **Backward rule (`bc_rule`)** | `use <goal> when <subgoals>`. Proves a goal by proving its subgoals (Prolog-style). |
| **`knowledge_engine.engine`** | The runtime object. `.activate(rb)` runs forward chaining; `.prove_goal(...)` runs backward chaining. |
| **`reset()` / `activate()`** | `reset()` clears asserted facts; `activate('rb')` loads a rule base and runs its forward rules. |
| **Plan** | Python code assembled during a successful backward proof — a callable returned alongside the variable bindings. |
| **`compiled_krb/`** | The cache directory where Pyke writes the `.py` it generates from your `.krb`/`.kfb`. |
| **Universal vs existential fact** | Pyke distinguishes facts that hold for all instances vs specific ones — relevant to how rules quantify. |

## 4. Setup

Pyke is **not on PyPI** in a form that installs on modern Python — it ships as a source
tarball on SourceForge (`pyke3` for Python 3), and its last release is from 2010. So unlike
most notebooks here, there is no reliable `pip install pyke` on Python 3.13.

```bash
# Conceptual install (Python 3 fork), from the SourceForge download:
#   https://sourceforge.net/projects/pyke/files/pyke/1.1.1/
# untar, then:
python setup.py build && python setup.py install   # legacy distutils
```

The cell below tries to import Pyke and records availability. Every live-engine cell is gated
behind `HAVE_PYKE`, so the notebook executes top-to-bottom whether or not Pyke is installed.
A small pure-Python chainer (Example 3) reproduces the core idea with real output regardless.

In [ ]:
# Pyke is a SourceForge-only, Python-2-era project; importing it is best-effort.
try:
    from pyke import knowledge_engine
    HAVE_PYKE = True
    print('Pyke is importable — live engine cells will run.')
except Exception as exc:  # ModuleNotFoundError on modern Python is expected
    HAVE_PYKE = False
    print('Pyke not installed (expected on Python 3.13):', type(exc).__name__)
    print('Gated cells will show the real .krb/.kfb syntax and call shape instead.')

## 5. Worked Examples

### Example 1 — the real Pyke artifacts: a fact base + a backward rule

Pyke's knowledge lives in text files. Below are the two files you'd write for a classic
family-relations knowledge base, and the Python that drives them. We always print the file
contents (the real Pyke DSL), and only *run* the engine when Pyke is importable.

In [ ]:
import os, tempfile, textwrap

# A fact base: base name `family`, then ground facts. Saved as family.kfb
family_kfb = textwrap.dedent('''\
    family

    parent_of(thomas, bruce)
    parent_of(mary, thomas)
    parent_of(helen, mary)
''')

# A rule base with a backward-chaining rule for grandparent, and a
# recursive ancestor rule. Saved as family_rules.krb
family_krb = textwrap.dedent('''\
    # grandparent($gp, $gc) holds when $gp is a parent of someone who
    # is a parent of $gc.
    bc_grandparent
        use grandparent($gp, $gc)
        when
            family.parent_of($gp, $p)
            family.parent_of($p, $gc)

    # ancestor is the transitive closure of parent_of (two rules).
    bc_ancestor_direct
        use ancestor($a, $d)
        when
            family.parent_of($a, $d)

    bc_ancestor_step
        use ancestor($a, $d)
        when
            family.parent_of($a, $mid)
            ancestor($mid, $d)
''')

print('=== family.kfb ===');       print(family_kfb)
print('=== family_rules.krb ==='); print(family_krb)

In [ ]:
# Drive the engine — only if Pyke is importable. This is the real call shape.
if HAVE_PYKE:
    src = tempfile.mkdtemp(prefix='pyke_kb_')
    with open(os.path.join(src, 'family.kfb'), 'w') as f:
        f.write(family_kfb)
    with open(os.path.join(src, 'family_rules.krb'), 'w') as f:
        f.write(family_krb)

    engine = knowledge_engine.engine(src)   # compiles .kfb/.krb -> compiled_krb/
    engine.reset()
    engine.activate('family_rules')         # load the rule base

    print('grandparents:')
    with engine.prove_goal('family_rules.grandparent($gp, $gc)') as gen:
        for vs, plan in gen:                # backward chaining; plan is None here
            print(' ', vs['gp'], '->', vs['gc'])
else:
    print('[skipped — Pyke not installed]')
    print('Call shape:')
    print("  engine = knowledge_engine.engine(src_dir)")
    print("  engine.reset(); engine.activate('family_rules')")
    print("  with engine.prove_goal('family_rules.grandparent($gp, $gc)') as gen:")
    print("      for vs, plan in gen: ...   # vs['gp'], vs['gc'] are the bindings")

### Example 2 — forward chaining and the plan-generation idea

Forward rules use `foreach ... assert ...`. The snippet below (shown as text — this is exactly
what a `.krb` would contain) derives `grandparent` facts eagerly from `parent_of`, instead of
proving on demand. The second snippet sketches Pyke's signature feature: a backward rule that
**assembles a plan** — the `with` block holds the Python that runs when the proof's plan is
later called.

In [ ]:
fc_rules = textwrap.dedent('''\
    # FORWARD chaining: when these premises match, assert the conclusion.
    fc_grandparent
        foreach
            family.parent_of($gp, $p)
            family.parent_of($p, $gc)
        assert
            family.grandparent_of($gp, $gc)
''')

plan_rule = textwrap.dedent('''\
    # PLAN generation: the proof of make_greeting assembles callable Python.
    # The `with` step becomes a function returned alongside the bindings.
    bc_greet
        use make_greeting($name)
        when
            person.likes($name, $thing)
        with
            print("Hello %s, enjoy your %s!" % ($name, $thing))
''')

print('=== forward-chaining rule ===');      print(fc_rules)
print('=== plan-generating backward rule ==='); print(plan_rule)
print('Run shape: engine.activate("fc_rules") fires fc_grandparent and stores')
print('new grandparent_of facts; prove_goal(...) on make_greeting returns a')
print('(vars, plan) pair where plan() executes the assembled `with` code.')

### Example 3 — a tiny pure-Python chainer (runs anywhere)

To *see* the inference Pyke automates, here is a ~25-line forward chainer over the same
family facts. It iterates the rules to a **fixpoint** (exactly what Pyke's `fc_*` rules do),
then we run a small **backward** query over the result. This executes with real output even
when Pyke is absent — it is a faithful miniature of the engine, not the engine itself.

In [ ]:
# Facts: parent_of(parent, child)
parent_of = {('thomas', 'bruce'), ('mary', 'thomas'), ('helen', 'mary')}

def forward_chain(parent_of):
    """Derive the full ancestor relation by firing rules to a fixpoint."""
    ancestor = set(parent_of)                      # rule 1: parent => ancestor
    while True:
        new = {(a, d) for (a, mid) in parent_of    # rule 2: parent + ancestor
                       for (m2, d) in ancestor if mid == m2}
        if new <= ancestor:                        # nothing new -> fixpoint
            return ancestor
        ancestor |= new

ancestors = forward_chain(parent_of)
print('all ancestor(a, d) facts derived by forward chaining:')
for a, d in sorted(ancestors):
    print(' ', a, '->', d)

# Backward-style query: prove grandparent(gp, gc) on demand.
def grandparents(parent_of):
    return {(gp, gc) for (gp, p) in parent_of
                     for (p2, gc) in parent_of if p == p2}

print('\ngrandparent(gp, gc):')
for gp, gc in sorted(grandparents(parent_of)):
    print(' ', gp, '->', gc)

The forward chainer keeps firing the recursive rule until it derives nothing new — that
fixpoint loop *is* forward chaining. The grandparent query joins two `parent_of` facts on the
shared middle person — that's a single backward rule body. Pyke does both for you over
arbitrary `.krb` rules, with proper unification, indexing, and (for backward proofs) plan
assembly on top.

## 6. Gotchas & Pitfalls

- **Installation is the hard part.** Pyke is unmaintained and SourceForge-only; there is no
  reliable `pip install` on modern Python. Expect to vendor the `pyke3` sources and fight
  `distutils`/`print`-statement-era code. This alone disqualifies it from most new projects.
- **It compiles to disk.** Pyke writes generated `.py` into a `compiled_krb/` directory on
  first run and reuses it. Stale caches bite: if you edit a `.krb` and behavior doesn't
  change, delete `compiled_krb/`. Also: the engine needs a real source *directory* to scan.
- **Forward vs backward is a deliberate choice per rule.** `fc_*` rules run when you
  `activate()` and eagerly assert facts; `bc_*` rules run only when you `prove_goal()`. Mixing
  them up (expecting a `bc_` rule to populate the KB on activation) is a classic confusion.
- **`$variables` vs Python.** Inside `.krb` files, `$x` are Pyke pattern variables, not Python
  names; they only get Python values after unification. The `with`/`assert` blocks are
  templated Python where `$x` is substituted.
- **`reset()` clears asserted facts.** Facts you assert at runtime vanish on `reset()`; only
  `.kfb` facts and rule-derived facts come back after re-activation. Order: `reset()` then
  `activate()`.
- **Plans must be *fully* grounded.** A plan is only useful if the proof bound every variable
  the `with` code references; an unbound `$var` in a plan is an error, not a free variable.
- **No termination guarantee.** Like Prolog, recursive backward rules can loop forever if
  written without a base case or with left-recursion; forward rules can fail to reach a
  fixpoint if a rule asserts ever-new facts.
- **Sparse, dated docs.** The tutorials assume Python 2 idioms. Treat examples as pseudo-code
  and verify against your installed version.

## 7. When to Use vs Alternatives

| Option | Paradigm | Use it when… | vs Pyke |
|--------|----------|--------------|---------|
| **Pyke** | Forward **and** backward chaining + plan generation | You specifically want plan/code assembly from proofs, or to study a dual-chaining Python engine | The only one here that emits callable plans; but unmaintained, hard to install |
| **SWI-Prolog / pyswip** (`swi-prolog`) | Backward chaining | Goal-directed queries, unification, mature & fast, real ecosystem | Prolog is the maintained, production-grade backward chainer; no plan generation, no `fc` rules |
| **CLIPS / clipspy** (`clips`) | Forward chaining (Rete) | Production rules over evolving facts; embeddable C engine, pip-installable | clipspy is maintained and fast; forward-only, no plans |
| **experta / pyknow** | Forward chaining (pure Python) | CLIPS-style rules with zero native deps | A maintained pure-Python alternative to Pyke's forward side |
| **Datalog** (`datalog`) | Bottom-up deduction | Pure, terminating recursive queries over relations | Decidable & side-effect-free; Pyke is more expressive but riskier |
| **miniKanren** (`minikanren`) | Relational logic | Embeddable, elegant relational programming in Python | Maintained, tiny, no file-based KB or plans |
| **Plain Python** | Imperative | A handful of fixed rules | No engine to install or learn |

**Rule of thumb:** for *new* work, you almost never pick Pyke — use pyswip for backward
chaining, clipspy/experta for forward chaining, or Datalog for pure deduction. Pick Pyke only
if its **plan-generation** model is exactly what you need (assembling Python call sequences
from declarative requirements) and you accept the maintenance cost, or you're studying it for
its historical and conceptual value.

## 8. Resources

- **Pyke official documentation (SourceForge project site)** — https://pyke.sourceforge.net/
- **Pyke downloads (source tarballs, incl. the Python 3 `pyke3` fork)** —
  https://sourceforge.net/projects/pyke/files/pyke/
- **Knowledge bases & rule syntax (`.kfb`/`.krb`) reference** —
  https://pyke.sourceforge.net/knowledge_bases/index.html
- **Plan generation tutorial (Pyke's signature feature)** —
  https://pyke.sourceforge.net/logic_programming/rules/index.html
- Cross-links in this library: `swi-prolog` (maintained backward chaining), `clips` and
  `experta`/`pyknow` (forward chaining), `datalog` (pure deduction), `minikanren` (relational
  logic), `rete-algorithm` (how forward matching is made fast).

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def prove(goal, facts, rules):
    """The plan assembled by a successful proof, or None when the goal fails."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE